In [ ]:
#!/usr/bin/env python3

import enum
import importlib.util
import json
import os
import re
import sys
import time
import warnings
from datetime import UTC, datetime, timezone
from glob import glob
from pathlib import Path

import django
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import scipy.integrate as integrate
import scipy.stats as st
import tqdm
from matplotlib import font_manager, rcParams
from scipy.spatial.transform import Rotation as rot

import HErmes as he
import HErmes.fitting as fit
import charmingbeauty as cb
import dashi as d
import gondola as gon


In [ ]:
# print(files)

paddles = gon.db.TofPaddle.all()
# test stand
# run_id = 10413
run_id = 251
# run_id = 10592
# run_id = 10029
# run_id = 144

numDataHitsHG = []

# data_path = '/home/gaps/csbf-data'
data_path = "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data"
dataset = Path(f"{data_path}/{run_id}")
files = [f for f in dataset.glob("*.tof.gaps")]


arrLTBtotLos = []
arrRBtotLos = []
arrRBvsRBnum = []
lowVetoHitCounts = []

numDataHitsLTB = []
numWFhits = []

chargeDataEv = [[], [], []]
chargeDataHit = [[], [], []]
numDataHits = [[], [], []]
paddleIdData = [[], [], []]
rbIdData = [[], [], []]

rbGmtbEvG = []
mtbGrbEvG = []
rbGmtbEvT = []
mtbGrbEvT = []

lowVeto = []
okVeto = []
vetoCord = []
testStandEv = []
mastEvList = []

rbIdDataLTB = []
paddleIdDataLTB = []
paddleIdDataHG = []
rbIdDataHG = []
rbIdDataRBEV = []

printFlag = False
onetime = False
timeArray48LTB = []
delTarr1 = []
evList = []

charge_a_arr = []
charge_b_arr = []

timingFromMaxA = []
timingFromMaxB = []

strSide = " "
indexId = 0
ltb_dict_store = {pdl.ltb_id: [] for pdl in paddles}
dataset = Path(f"{data_path}/{run_id}")
files = [f for f in dataset.glob("*.gaps")]
calib = gon.calibration.load_rb_calibrations(Path('/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/calib/latest'))

for f in files:  # [0:endF]
    print(f)
    reader = gon.io.TofPacketReader(
        str(f),
        filter=gon.packets.TofPacketType.TofEvent,
    )
    packcntEv = 0
    
    for pack in reader:
        #print("in")
        Qev = [0, 0, 0]
        numHits = [0, 0, 0]
        ev = gon.events.TofEvent()

        # print(pack)

        # evWF = go.events.RBWaveform()
        # evWF.from_tofpacket(pack)
        # evWF.calibrate(calib[wf.rb_id])
        # ev = go.rust_api.events.TofEvent()
        ev.from_tofpacket(pack)
        # evList.append(ev.event_id)
        packcntEv += 1

        # for wf in ev.waveforms:
        #     plt.plot(wf.adc_a, label=f"side a; paddle {wf.paddle_id}")
        #     plt.plot(wf.adc_b, label=f"side b; paddle {wf.paddle_id}")
        #     plt.legend()
        #     plt.show()
        for wf in ev.waveforms:
            if True:  # (wf.paddle_id == paddle_id_set)
                
                wf.calibrate(calib[wf.rb_id])
                print("here!")
                if np.size(wf.times_a) != 0:
                    # print(ev.mastertriggerevent.trigger_sources)
                    # print(max(wf.voltages_a[10:]))
                    start_idx = 10
                    sublist = wf.voltages_a[start_idx:]
    
                    if np.max(sublist) > 740:
                        # Get index of max in the sublist, then add start_idx
                        # to get the original index
                        max_indexa = np.argmax(sublist) + start_idx

                        sublist = wf.voltages_b[start_idx:]
                        max_indexb = np.argmax(sublist) + start_idx

                        timingFromMaxA.append(wf.times_a[max_indexa])
                        timingFromMaxB.append(wf.times_b[max_indexb])

                        plt.plot(
                            wf.times_a,
                            wf.voltages_a,
                            label=f"side a; paddle {wf.paddle_id}",
                        )
                        plt.plot(
                            wf.times_b,
                            wf.voltages_b,
                            label=f"side b; paddle {wf.paddle_id}",
                        )
                        plt.legend()
                        plt.show()
                        time.sleep(1)
            # print("end of wf for ev")

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import time
import gondola as gon



timingFromMaxA = []
timingFromMaxB = []



calib = gon.calibration.load_rb_calibrations(
    Path("/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/calib/latest")
)



run_id = 251
data_path = "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data"
dataset = Path(f"{data_path}/{run_id}")



# Prefer TOF packet files if present
files = list(dataset.glob("*.tof.gaps"))
print("nfiles =", len(files))



for f in files:
    print(f"Opening {f}")

    reader = gon.io.TofPacketReader(
        str(f),
        filter=gon.packets.TofPacketType.TofEvent,
    )

    packcntEv = 0

    for pack in reader:
        print("pack type:", type(pack))

        ev = gon.events.TofEvent()
        try:
            ev.from_tofpacket(pack)
        except Exception as e:
            print("from_tofpacket failed")
            print("pack =", pack)
            print("error =", e)
            raise

        packcntEv += 1
        print(f"event {packcntEv}: {ev}")

        for wf in ev.waveforms:
            print("waveform paddle:", wf.paddle_id, "rb:", wf.rb_id)

            try:
                wf.calibrate(calib[wf.rb_id])
            except Exception as e:
                print(f"calibration failed for rb_id={wf.rb_id}: {e}")
                continue

            if np.size(wf.times_a) == 0:
                continue

            start_idx = 10
            sublist_a = wf.voltages_a[start_idx:]

            if len(sublist_a) == 0:
                continue

            if np.max(sublist_a) > 740:
                max_indexa = np.argmax(sublist_a) + start_idx

                sublist_b = wf.voltages_b[start_idx:]
                if len(sublist_b) == 0:
                    continue

                max_indexb = np.argmax(sublist_b) + start_idx

                timingFromMaxA.append(wf.times_a[max_indexa])
                timingFromMaxB.append(wf.times_b[max_indexb])

                plt.plot(wf.times_a, wf.voltages_a, label=f"side a; paddle {wf.paddle_id}")
                plt.plot(wf.times_b, wf.voltages_b, label=f"side b; paddle {wf.paddle_id}")
                plt.legend()
                plt.show()

        break  # just first event for debugging

    break  # just first file for debugging

In [ ]:

%matplotlib inline

from pathlib import Path
import time

import numpy as np
import matplotlib.pyplot as plt
import gondola as gon

# ---------------------------------
# setup
# ---------------------------------
timingFromMaxA = []
timingFromMaxB = []
numDataHitsHG = []

run_id = 251
# run_id = 10592
# run_id = 10029   # teststand 2025
# run_id = 144

data_path = "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data"
dataset = Path(f"{data_path}/{run_id}")
files = list(dataset.glob("*.tof.gaps"))

print("nfiles =", len(files))

calib = gon.calibration.load_rb_calibrations(
    Path("/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/calib/251123_215129UTC")
)
print("n calib entries =", len(calib))

# ---------------------------------
# loop over files / events
# ---------------------------------
endF = 20
for f in files[:endF]:  # or files[:endF]
    print("opening:", f)

    reader = gon.io.TofPacketReader(str(f))
    packcntEv = 0

    for pack in reader:
        # skip packets that are not actual TofEvent packets
        if pack.packet_type != gon.packets.TofPacketType.TofEvent:
            continue

        Qev = [0, 0, 0]
        numHits = [0, 0, 0]

        # IMPORTANT:
        # do NOT use ev.from_tofpacket(pack) here
        # for these files, the correct path is from_bytestream(pack.payload, 0)
        ev = gon.events.TofEvent.from_bytestream(pack.payload, 0)
        time.sleep(1)
        packcntEv += 1
        numDataHitsHG.append(len(ev.hits))
        # optional slow-down for interactive debugging
        for rb in ev.rb_events:
            for hit in rb.hits:
                hit.charge_a
                hit.charge_b


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

charges = []

endF = 30
for f in files[:endF]:
    print("opening:", f)

    reader = gon.io.TofPacketReader(str(f))

    for pack in reader:
        if pack.packet_type != gon.packets.TofPacketType.TofEvent:
            continue

        ev = gon.events.TofEvent.from_bytestream(pack.payload, 0)

        for rb in ev.rb_events:
            for hit in rb.hits:
                charges.append(hit.charge_a)
                charges.append(hit.charge_b)

# Convert to numpy array
charges = np.array(charges)

# Optional: filter range (avoid crazy outliers)
charges = charges[(charges >= 0) & (charges <= 1000)]

# Plot histogram

In [ ]:
plt.hist(charges, bins=100, range=(0, 1000))
plt.xlabel("Charge (pC)")
plt.ylabel("Counts")
plt.yscale("log")
plt.title("Charge Distribution (A & B)")
plt.show()